# Implementação de modelo de Machine Learning com dados do Campeonato Brasileiro Série A 

Esse notebook constitui a implementação do Modelo de ML para o trabalho prático da disciplina "Introdução à Ciência dos Dados".

**Alunos**:

- Lucas de Oliveira Ferreira 
- Caio Henrique de Miranda Onofre
- João Pedro Coutinho Capelão


### Imports e configurações

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.pipeline import Pipeline


# Configurações globais de plotagem
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## Leitura e filtro do Dataset

In [2]:
df_raw = pd.read_csv('../data/mundo_transfermarkt_competicoes_brasileirao_serie_a.csv')
df_raw.head()

,ano_campeonato,data,rodada,estadio,arbitro,publico,publico_max,time_mandante,time_visitante,tecnico_mandante,...,chutes_bola_parada_mandante,chutes_bola_parada_visitante,defesas_mandante,defesas_visitante,impedimentos_mandante,impedimentos_visitante,chutes_mandante,chutes_visitante,chutes_fora_mandante,chutes_fora_visitante
0,2005,2005-10-08,31,Estádio Durival Britto e Silva,NaN,NaN,NaN,Paraná,Fluminense,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2005,2005-04-23,1,Estádio Durival Britto e Silva,NaN,NaN,NaN,Paraná,Goiás EC,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2005,2005-08-21,21,Estádio Durival Britto e Silva,NaN,NaN,NaN,Paraná,Vasco da Gama,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2005,2005-08-28,23,Estádio Durival Britto e Silva,NaN,NaN,NaN,Paraná,São Paulo,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2005,2005-06-26,9,Estádio Olímpico Nilton Santos,NaN,NaN,NaN,Botafogo,Figueirense FC,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Removemos dados depois de 2020 e antes de 2023. A razão desse crop se encontra no outro notebook.

In [3]:
df_raw = df_raw[(df_raw['ano_campeonato'] >= 2020) & (df_raw['ano_campeonato'] <= 2023)].copy()
df_raw = df_raw.dropna(subset=['chutes_mandante'])

Agora, construimos o dataset com as features que serão utilizadas no modelo.

In [4]:

# 1. Garantir a ordenação 
df_model = df_raw.sort_values(by=['ano_campeonato', 'data', 'rodada']).copy()

# 2. Criar as médias móveis de CHUTES
df_model['media_chutes_mandante_5j'] = df_model.groupby('time_mandante')['chutes_mandante'].shift(1).rolling(window=5, min_periods=1).mean()
df_model['media_chutes_visitante_5j'] = df_model.groupby('time_visitante')['chutes_visitante'].shift(1).rolling(window=5, min_periods=1).mean()

# 3. Criar as médias móveis de GOLS 
df_model['media_gols_mandante_5j'] = df_model.groupby('time_mandante')['gols_mandante'].shift(1).rolling(window=5, min_periods=1).mean()
df_model['media_gols_visitante_5j'] = df_model.groupby('time_visitante')['gols_visitante'].shift(1).rolling(window=5, min_periods=1).mean()

# 4. Calcular a Eficiência
df_model['eficiencia_mandante_5j'] = np.where(
    df_model['media_chutes_mandante_5j'] > 0, 
    (df_model['media_gols_mandante_5j'] / df_model['media_chutes_mandante_5j']) * 100, 
    0
)

df_model['eficiencia_visitante_5j'] = np.where(
    df_model['media_chutes_visitante_5j'] > 0, 
    (df_model['media_gols_visitante_5j'] / df_model['media_chutes_visitante_5j']) * 100, 
    0
)

# 5. Fazemos as variáveis virarem diferenças
df_model['dif_valor_titular'] = df_model['valor_equipe_titular_mandante'] - df_model['valor_equipe_titular_visitante']
df_model['dif_eficiencia'] = df_model['eficiencia_mandante_5j'] - df_model['eficiencia_visitante_5j']

# 6. Limpeza das rodadas sem histórico
df_model = df_model.dropna(subset=[
    'media_chutes_mandante_5j', 'media_chutes_visitante_5j', 
    'eficiencia_mandante_5j', 'eficiencia_visitante_5j', 
    'dif_valor_titular'
])

Agora enfim, vamos treinar o modelo:

In [5]:
# 1. Definir features
features_finais = [
    'media_chutes_mandante_5j', 'media_chutes_visitante_5j', 
    'eficiencia_mandante_5j', 'eficiencia_visitante_5j', 
    'dif_valor_titular'
]

X = df_model[features_finais]
y = df_model['gols_mandante'].astype(int) - df_model['gols_visitante'].astype(int) # Variável que vamos prever

# 2. Split Temporal (Treino até 2022, Teste para 2023)
treino_mask = df_model['ano_campeonato'] <= 2022
X_train, X_test = X[treino_mask], X[~treino_mask]
y_train, y_test = y[treino_mask], y[~treino_mask]

# 3. Construção do Pipeline Robusto
# O Pipeline garante que o StandardScaler seja recalculado a cada iteração da validação cruzada,
# usando apenas os dados disponíveis até aquele momento.
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor(weights='distance'))
])

# 4. Definir a grade de busca (Vamos testar K ímpares de 3 a 31 para evitar empates)
param_grid = {'knn__n_neighbors': np.arange(3, 33, 2)}

# 5. Validação Cruzada Temporal (5 janelas de tempo progressivas)
tscv = TimeSeriesSplit(n_splits=5)

# 6. Executar a Otimização
# scoring em neg_mean_absolute_error pois o GridSearch sempre tenta maximizar a métrica
grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    cv=tscv, 
    scoring='neg_mean_absolute_error', 
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

# 7. Resultados do Treinamento
melhor_k = grid_search.best_params_['knn__n_neighbors']
melhor_mae_cv = -grid_search.best_score_

print(f"Otimização concluída!")
print(f"Melhor valor de K encontrado: {melhor_k} vizinhos")
print(f"MAE esperado (baseado na validação temporal): {melhor_mae_cv:.3f} gols\n")

# 8. O Teste Final (A Prova de Fogo em 2023)
modelo_final = grid_search.best_estimator_
previsoes_finais = modelo_final.predict(X_test)
mae_teste = mean_absolute_error(y_test, previsoes_finais)

print("--- RESULTADO FINAL NO TESTE (2023) ---")
print(f"MAE Final: {mae_teste:.3f} gols")

Otimização concluída!
Melhor valor de K encontrado: 31 vizinhos
MAE esperado (baseado na validação temporal): 1.143 gols

--- RESULTADO FINAL NO TESTE (2023) ---
MAE Final: 1.251 gols


Vamos calcular o intervalo de confiança via Bootstrapping. 

In [6]:
y_test_array = np.array(y_test)
previsoes_finais = modelo_final.predict(X_test) # modelo_final vem do grid_search.best_estimator_

# ==========================================
# CÁLCULO DO INTERVALO DE CONFIANÇA (BOOTSTRAPPING)
# ==========================================
n_iterations = 1000
maes_bootstrapped = []
n_size = len(y_test_array)

np.random.seed(42) # Mantém os resultados consistentes caso rode a célula de novo
for i in range(n_iterations):
    indices = np.random.randint(0, n_size, n_size)
    amostra_y = y_test_array[indices]
    amostra_prev = previsoes_finais[indices]
    maes_bootstrapped.append(mean_absolute_error(amostra_y, amostra_prev))

ic_inferior = np.percentile(maes_bootstrapped, 2.5)
ic_superior = np.percentile(maes_bootstrapped, 97.5)
mae_teste = mean_absolute_error(y_test, previsoes_finais)

print("--- 1. QUALIDADE GERAL DO MODELO (DADOS DE 2023) ---")
print(f"MAE Final: {mae_teste:.3f} gols")
print(f"Intervalo de Confiança (95%): [{ic_inferior:.3f}, {ic_superior:.3f}] gols\n")

--- 1. QUALIDADE GERAL DO MODELO (DADOS DE 2023) ---
MAE Final: 1.251 gols
Intervalo de Confiança (95%): [1.152, 1.350] gols



In [7]:
# ==========================================
# 1. PREVISÃO EM UM JOGO REAL
# ==========================================
# Primeiro jogo do Cruzeiro como mandante no ano de teste
jogos_teste = df_model[~treino_mask]
jogo_exemplo = jogos_teste[jogos_teste['time_mandante'] == 'Cruzeiro'].iloc[0:1]

mandante = jogo_exemplo['time_mandante'].values[0]
visitante = jogo_exemplo['time_visitante'].values[0]
saldo_real = int(jogo_exemplo['gols_mandante'].values[0]) - int(jogo_exemplo['gols_visitante'].values[0])

# Separando apenas as features que o modelo precisa
X_exemplo = jogo_exemplo[features_finais]

# O modelo_final (Pipeline) já aplica a padronização e faz a predição automaticamente
saldo_previsto = modelo_final.predict(X_exemplo)[0]

print(f"---  JOGO ALVO: {mandante} x {visitante} ---")
print(f"Resultado Real do Jogo: Saldo de {saldo_real} gol(s)")
print(f"Predição do Modelo (Saldo Esperado): {saldo_previsto:.2f} gols\n")

# ==========================================
# 2. EXPLICABILIDADE: QUEM SÃO OS VIZINHOS?
# ==========================================
# Como usamos um Pipeline, precisamos "desempacotar" o scaler e o KNN para usar a função kneighbors
scaler_opt = modelo_final.named_steps['scaler']
knn_opt = modelo_final.named_steps['knn']

# Precisamos passar os dados do jogo pelo scaler isolado
X_exemplo_scaled = scaler_opt.transform(X_exemplo)

# Buscando os K vizinhos com o KNN otimizado
distancias, indices_vizinhos = knn_opt.kneighbors(X_exemplo_scaled)
print("\n--------------------------------------------------------------------------------------------------\n")
print("---  HISTÓRICO: OS JOGOS MAIS PARECIDOS ENCONTRADOS ---")
# Usamos os índices para resgatar os jogos originais na base de treino
df_treino = df_model[treino_mask]
vizinhos = df_treino.iloc[indices_vizinhos[0]]

resumo_vizinhos = vizinhos[['data', 'time_mandante', 'time_visitante', 'gols_mandante', 'gols_visitante']].copy()
resumo_vizinhos['saldo_historico'] = resumo_vizinhos['gols_mandante'].astype(int) - resumo_vizinhos['gols_visitante'].astype(int)

# Exibe os 5 primeiros vizinhos encontrados para justificar a nota
print(resumo_vizinhos.head())

---  JOGO ALVO: Cruzeiro x Grêmio ---
Resultado Real do Jogo: Saldo de 1 gol(s)
Predição do Modelo (Saldo Esperado): 0.35 gols


--------------------------------------------------------------------------------------------------

---  HISTÓRICO: OS JOGOS MAIS PARECIDOS ENCONTRADOS ---
            data  time_mandante time_visitante  gols_mandante  gols_visitante  \
4178  2022-04-10      Fortaleza      Cuiabá-MT            0.0             1.0   
5113  2021-07-04       Ceará SC      Juventude            2.0             0.0   
5383  2021-10-22  Internacional  RB Bragantino            1.0             1.0   
3964  2022-10-06       Ceará SC          Goiás            1.0             1.0   
5552  2020-12-06    Coritiba FC  RB Bragantino            0.0             0.0   

      saldo_historico  
4178               -1  
5113                2  
5383                0  
3964                0  
5552                0  


# Interface gráfica (extra)

Dado que o modelo consegue estimar o resultado de partidas, podemos repitir o teste mas com uma interface gráfica um pouco mais atraente e interativa. Utilizando ipywidgets, podemos fazer isso. 

In [9]:
import ipywidgets as widgets
from IPython.display import display

# 1. Preparação dos dados de teste
jogos_teste = df_model[~treino_mask].copy()
times_unicos = sorted(jogos_teste['time_mandante'].unique())

# 2. Componentes de Entrada
dropdown_mandante = widgets.Dropdown(
    options=times_unicos,
    description='Mandante:',
    layout={'width': '250px'}
)

dropdown_visitante = widgets.Dropdown(
    options=[], 
    description='Visitante:',
    layout={'width': '250px'}
)

btn_simular = widgets.Button(
    description='Simular Previsão',
    button_style='primary',
    icon='soccer-ball-o',
    layout={'margin': '0px 0px 0px 20px'}
)

# 3. Componentes de Saída 
# Ambos agora são widgets HTML puros. Zero chance de duplicar!
texto_resultado = widgets.HTML(value="") 
tabela_html = widgets.HTML(value="") # <-- Mudança aqui!

# 4. Lógica de Atualização Dinâmica
def atualizar_visitantes(*args):
    mandante_atual = dropdown_mandante.value
    adversarios = sorted(jogos_teste[jogos_teste['time_mandante'] == mandante_atual]['time_visitante'].unique())
    dropdown_visitante.options = adversarios

dropdown_mandante.observe(atualizar_visitantes, 'value')
atualizar_visitantes()

# 5. Função Principal
def simular(b):
    mandante_selecionado = dropdown_mandante.value
    visitante_selecionado = dropdown_visitante.value
    
    filtro = (jogos_teste['time_mandante'] == mandante_selecionado) & (jogos_teste['time_visitante'] == visitante_selecionado)
    jogo_exemplo = jogos_teste[filtro]
    
    if jogo_exemplo.empty:
        texto_resultado.value = "<b style='color:red;'>⚠️ Jogo não encontrado na base de teste.</b>"
        tabela_html.value = ""
        return
        
    jogo_exemplo = jogo_exemplo.iloc[0:1]
    
    mandante = jogo_exemplo['time_mandante'].values[0]
    visitante = jogo_exemplo['time_visitante'].values[0]
    gols_mandante = int(jogo_exemplo['gols_mandante'].values[0])
    gols_visitante = int(jogo_exemplo['gols_visitante'].values[0])
    saldo_real = gols_mandante - gols_visitante
    
    # Predição
    X_exemplo = jogo_exemplo[features_finais]
    saldo_previsto = modelo_final.predict(X_exemplo)[0]
    
    # Explicabilidade
    scaler_opt = modelo_final.named_steps['scaler']
    knn_opt = modelo_final.named_steps['knn']
    
    X_exemplo_scaled = scaler_opt.transform(X_exemplo)
    distancias, indices_vizinhos = knn_opt.kneighbors(X_exemplo_scaled)
    
    df_treino = df_model[treino_mask]
    vizinhos = df_treino.iloc[indices_vizinhos[0]].copy()
    vizinhos['saldo'] = vizinhos['gols_mandante'].astype(int) - vizinhos['gols_visitante'].astype(int)
    
    # Montando o HTML do texto principal
    html_content = f"""
    <hr style="border: 1px dashed #ccc;">
    <h3 style="margin-bottom: 5px;">🏟️ JOGO: {mandante} {gols_mandante} x {gols_visitante} {visitante}</h3>
    <p style="margin-top: 0; font-size: 14px;">
        <b>📊 SALDO REAL:</b> {saldo_real} gol(s)<br>
        <b>🤖 PREVISÃO DO MODELO:</b> {saldo_previsto:.2f} gols de saldo
    </p>
    <hr style="border: 1px dashed #ccc;">
    <h4 style="margin-bottom: 5px;">🔍 POR QUE O MODELO PREVIU ISSO?</h4>
    <p style="margin-top: 0;">Estes foram os 5 jogos mais parecidos encontrados no histórico (2020-2022):</p>
    """
    texto_resultado.value = html_content
    
    # ==========================================
    # SOLUÇÃO DA TABELA: Renderizamos o Styler do Pandas direto para HTML
    # ==========================================
    tabela_estilizada = (vizinhos[['data', 'time_mandante', 'time_visitante', 'gols_mandante', 'gols_visitante', 'saldo']]
                         .head(5)
                         .rename(columns={'saldo': 'Saldo Final'})
                         .style.hide(axis='index')
                         .background_gradient(cmap='RdYlGn', subset=['Saldo Final']))
    
    # O método .to_html() converte a tabela do pandas numa string de código HTML
    tabela_html.value = tabela_estilizada.to_html()

btn_simular.on_click(simular)

# 6. Exibir Layout Organizado
layout_interativo = widgets.VBox([
    widgets.HTML("<h2>🎮 Simulador de Previsões KNN - Brasileirão</h2>"),
    widgets.HBox([dropdown_mandante, dropdown_visitante, btn_simular]),
    texto_resultado, 
    tabela_html      # Renderiza a string HTML da tabela
])

display(layout_interativo)